# Creating MCP Server for Tools

## Step 1: Setting up the MCP Server

In [ ]:
# All Necessary Imports

%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets


import sys
sys.path.append('../05_src/')


from pydantic import BaseModel, Field
from typing import List, Dict
from langchain.tools import tool
from utils.clients import get_client
from helpers.templates import line_graph, pie_chart, bar_graph 

import os
import requests
import json


MODEL=os.getenv("MODEL")






The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


#### Loading the client

In [5]:
client = get_client()

In [4]:

from fastmcp import FastMCP

mcp = FastMCP(
    name="can_gov_server",
    instructions="""
    This server provides visual responses to user's queries by fetching statistics on the official canadian government statistics API and returns a graph in the form of python code. 
    """
)

#### Tools description

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_stats_from_govcan",
        "description": "Get data from the Canadian government statistics on a specific topic within a specific time period.",
        "parameters": {
            "type": "object",
            "properties": {
                 "type": {
                    "type": "string",
                    "description": "Appropriate type of graph for data. Can only be line, bar, or pie chart.",
                },
                "title": {
                    "type": "string",
                    "description": "Title for a graph based on the given data.",
                },
                 "xaxis": {
                    "type": "string",
                    "description": "Title for the independent variable given the information.",
                },
                 "yaxis": {
                    "type": "string",
                    "description": "Title for the dependent variable given the information.",
                },
                 "xdata": {
                    "type": "list",
                    "description": "Data relating to the independent variable",
                },
                "ydata": {
                    "type": "list",
                    "description": "Data relating to the dependent variable",
                }
            },
            "required": ["title", "xaxis", "yaxis", "xdata", "ydata"],
            "additionalProperties": False,
        },
        "strict": True,
    },
        {
        "type": "function",
        "name": "create_graph",
        "description": "Generate a graph in Python with the given information regarding graph type, title, axis titles, axis data",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": f"""
                        Using the provided data, generate code in Python.

                        Follow the template for a line graph: {line_graph}
                        
                        Follow the template for a bar graph: {bar_graph}

                        Follow the template for a pie chart: {pie_chart}
                        """,
                }
            },
            "required": ["code"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

## Step 2: Adding Graph Data Tool

In [6]:
@mcp.tool

def get_stats_from_govcan(product_id=35100003, n_latest=25):
    """
    An API call to a Canadian government statistics service is made.
    The API call is to hhttps://www150.statcan.gc.ca/t1/wds/rest/getDataFromCubePidCoordAndLatestNPeriods
    and takes two parameters product_id (the topic of the response) and n_lastest (the number of reports that should be included in the response).
    Accepted values for product_id are: 8-10 digits
    Accepted values for n_latest are: 0-1000.
    """
        
    res = get_data_from_service(product_id,n_latest)
    data = get_data_from_response(res)
    return data
    
def get_data_from_service(product_id, n_latest):

    endpoint_api_url="https://www150.statcan.gc.ca/t1/wds/rest/getDataFromCubePidCoordAndLatestNPeriods"
    params=[{
        "productId": product_id, 
        "coordinate": "1.12.0.0.0.0.0.0.0.0", 
        "latestN":n_latest}]
    
    response = requests.post(endpoint_api_url, json=params)
    return response

def get_data_from_response(response) -> Dict[str, List]:

    vector = json.loads(response.text)

    x_data=[]
    y_data=[]

    points=vector[0]["object"]["vectorDataPoint"]

    for point in points:
        x_data.append(point["refPer2"])
        y_data.append(point["value"])

    return {"x": x_data, "y": y_data}



## Step 3: Adding Graph Generator Tool

In [7]:
@mcp.tool
def create_graph(title="", data=[]):
    """
    An invoke request is made to gpt-40-mini.
    The call takes two parameters title (the title of the graph) and data (the data points for the graph).
    """
    STYLE_NOTES = """
    Styling conventions to match:
    - Colors: PALETTE = ["#4C6EF5", "#F76707", "#12B886", "#FAB005", "#E64980", "#7048E8"]
    - Font: Helvetica Neue/Arial, size 14, color #2B2B2B; title size 22, left-aligned
    - Line charts: spline curves, width 3, markers size 9 with white outline
    - Bar charts: colored by category, labels outside bars, no border, bargap 0.35
    - Pie charts: donut style (hole=0.45), labels outside, slight slice separation
    """

    instruction = f"""
    You are a senior developer and expert statistician. You will be given data and a chart title.

    Three helper functions already exist in the target environment: line_chart(data, x_col, y_col, title, x_label, y_label), bar_chart(data, x_col, y_col, title, x_label, y_label), and pie_chart(data, names_col, values_col, title). Do NOT redefine these functions or restate their bodies — they already apply the styling described below internally.

    {STYLE_NOTES}

    Your job is only to:
    1. Build a `data` dict from the values provided.
    2. Transform data types if necessary or if suitable.
    3. Call the correct chart function with appropriate x_col/y_col (or names_col/values_col), title, and axis labels inferred from the data.
    4. Write a full python code with all the necessary imports, function, comments, and styling. 
    5. Call .show() on the result.

    Output ONLY the Python code for steps 1-5, in a single code block.
    """

    prompt=f"Write a Python code for a graph titled {title}, using these data: {data}"

    response = client.responses.create(
        model=MODEL,
        instructions=instruction,
        input=[{'role': 'user', 'content': prompt}],
        max_output_tokens=3000,
        temperature=1
    )

    code = response.output[0].content[0].text
    return code


## Step 5: Deploying

In [10]:
mcp.run(
        transport="http",
        host="localhost", 
        port=3008, 
    )


RuntimeError: Already running asyncio in this thread